In [2]:

# import pandas as pd
# import numpy as np
# from numpy.linalg import norm

# df = pd.read_csv('course_ratings.csv')

# rating_matrix = df.pivot(index='userId', columns='courseId', values='rating').fillna(0)

# def cosine_similarity(v1, v2):
#     if norm(v1) == 0 or norm(v2) == 0:
#         return 0
#     return np.dot(v1, v2) / (norm(v1) * norm(v2))

# course_titles = {
#     "cm8u9l665000csprox9w7z2f9": "Web Development 101",
#     "cm8u9l65y000asproacv1c3c7": "Python for Beginners",
#     "cm8u9l65r0008spro8d81clgm": "Data Science Essentials",
#     "cm8u9l65e0006sproli0k8mua": "UI/UX Design Basics"
# }

# recommendations = {}

# for target_user in rating_matrix.index:
#     target_vector = rating_matrix.loc[target_user].values

#     similarities = {}
#     for other_user in rating_matrix.index:
#         if other_user != target_user:
#             sim = cosine_similarity(target_vector, rating_matrix.loc[other_user].values)
#             similarities[other_user] = sim

#     similar_users = sorted(similarities.items(), key=lambda x: x[1], reverse=True)

#     course_scores = {}
#     for other_user, similarity in similar_users:
#         for course in rating_matrix.columns:
#             if rating_matrix.loc[target_user, course] == 0:  # course not rated by target user
#                 course_scores.setdefault(course, 0)
#                 course_scores[course] += similarity * rating_matrix.loc[other_user, course]

#     top_courses = sorted(course_scores.items(), key=lambda x: x[1], reverse=True)[:4]
#     recommendations[target_user] = top_courses

# print("\n📚 Course Recommendations:")
# for user_id, recs in recommendations.items():
#     print(f"\n👤 User {user_id}:")
#     if recs:
#         for course_id, score in recs:
#             title = course_titles.get(course_id, course_id)  # fallback to course_id if title not found
#             print(f"  - {title} (score: {score:.2f})")
#     else:
#         print("  No recommendations available.")









import pandas as pd
import numpy as np
from numpy.linalg import norm
import mysql.connector

# Step 1: Connect to MySQL
connection = mysql.connector.connect(
    host='localhost',
    user='your_username',
    password='your_password',
    database='your_database_name'
)

# Step 2: Fetch data
query = "SELECT userId, courseId, rating FROM ratings"
df = pd.read_sql(query, connection)

# Step 3: Pivot to Rating Matrix
rating_matrix = df.pivot(index='userId', columns='courseId', values='rating').fillna(0)

# Step 4: Cosine Similarity Function
def cosine_similarity(v1, v2):
    if norm(v1) == 0 or norm(v2) == 0:
        return 0
    return np.dot(v1, v2) / (norm(v1) * norm(v2))

# Step 5: Fetch course titles
course_query = "SELECT id, title FROM courses"
course_df = pd.read_sql(course_query, connection)
course_titles = dict(zip(course_df['id'], course_df['title']))

# Step 6: Recommendation Logic
recommendations = {}

for target_user in rating_matrix.index:
    target_vector = rating_matrix.loc[target_user].values

    similarities = {}
    for other_user in rating_matrix.index:
        if other_user != target_user:
            sim = cosine_similarity(target_vector, rating_matrix.loc[other_user].values)
            similarities[other_user] = sim

    similar_users = sorted(similarities.items(), key=lambda x: x[1], reverse=True)

    course_scores = {}
    for other_user, similarity in similar_users:
        for course in rating_matrix.columns:
            if rating_matrix.loc[target_user, course] == 0:  # course not rated by target user
                course_scores.setdefault(course, 0)
                course_scores[course] += similarity * rating_matrix.loc[other_user, course]

    top_courses = sorted(course_scores.items(), key=lambda x: x[1], reverse=True)[:4]
    recommendations[target_user] = top_courses

# Step 7: Display Output
print("\n📚 Course Recommendations:")
for user_id, recs in recommendations.items():
    print(f"\n👤 User {user_id}:")
    if recs:
        for course_id, score in recs:
            title = course_titles.get(course_id, course_id)
            print(f"  - {title} (score: {score:.2f})")
    else:
        print("  No recommendations available.")
